# Primary Import and Export Countries by State

Following the announcement by the new administration to raise tariffs on Canada, Mexico, and China (and the not entirely unexpected retaliatory measures from those targeted countries), it would be interesting to know which countries are the primary sources and destinations for imports and exports.

This data can be obtained from the US government’s International Trade Administration at trade.gov however, it is not easily obtained since the APIs made available to the public do not have any endpoints to access this type of data. Instead, we will have to resort to scraping the data from where it is available – a web page located at:

[https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProduct?%3AshowVizHome=no&%3Aembed=true&%3Atoolbar=yes%2Ctop](https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProduct?%3AshowVizHome=no&%3Aembed=true&%3Atoolbar=yes%2Ctop)

As is often the case with scraping web pages, the data is buried deep within the structure making it unfeasible to use quick and dirty methods like grabbing a table from a webpage using native Pandas functionality. Furthermore, the data on this page is displayed only a single state at a time and requires interacting with the webpage to modify filters. Selenium to the rescue!

After acquiring the data, we also want to display it graphically using a map of the USA to show each state’s dependence on their primary export and import partners. For this, we will employ Choropleth maps.

In [1]:
import requests

api_key = "ENTERYOURAPIKEYHERE"

In [2]:
# Some configurations required for the remainder of the script

DATA_YEAR = 2025

# From previous runs, we know we need a palette of 19 colors. Generate the palette using:
# https://medialab.github.io/iwanthue/
# and set the Color Space to full range of H, C, but for L set the range from 50.0 to 100.0
# Cycle through the color by rerolling the palette and locking in those you like
# Then take the 19 colors generated and sort them by pasting into the Colors box in
# https://onlinepngtools.com/sort-colors
# Take the sorted colors and apply them to the dictionary keys (Countries) below

SHARED_COLOR_MAP = {
      "Australia":      "#9f202c"
    , "Brazil":         "#ff6b9f"
    , "Canada":         "#d20a98"
    , "China":          "#9b0fa1"
    , "Costa Rica":     "#bba5ff"
    , "Germany":        "#553abb"
    , "India":          "#3c4aa7"
    , "Ireland":        "#8fbdff"
    , "Italy":          "#0170b6"
    , "Japan":          "#01de74"
    , "Malaysia":       "#1a6000"
    , "Mexico":         "#b8ce94"
    , "Netherlands":    "#8ec116"
    , "Saudi Arabia":   "#e4c44f"
    , "South Korea":    "#d99e00"
    , "Switzerland":    "#b16200"
    , "Taiwan":         "#ff9c30"
    , "Thailand":       "#906441"
    , "United Kingdom": "#d95105"
}

The following attempt fails because even though the variables used come from the documented set of variables at [https://api.census.gov/data/timeseries/intltrade/exports/enduse/variables.html](https://api.census.gov/data/timeseries/intltrade/exports/enduse/variables.html) and otherwise accessing the API directly as an https query using [https://api.census.gov/data/timeseries/intltrade/exports/enduse?get=ALL_VAL_YR,DISTRICT,DIST_NAME&time=2023&key=YOUR_API_KEY_HERE](https://api.census.gov/data/timeseries/intltrade/exports/enduse?get=ALL_VAL_YR,DISTRICT,DIST_NAME&time=2023&key=YOUR_API_KEY_HERE), the programmatic API call fails. In any case, it doesn't provide the state by state export totals nor the destination countries.

So instead, we will use selenium with chromedriver to access the page located at [https://www.trade.gov/data-visualization/tradestats-express-state-trade-product](https://www.trade.gov/data-visualization/tradestats-express-state-trade-product) although that page is simply a container for n iframe whose contents are actually located at [https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProduct?%3AshowVizHome=no&%3Aembed=true&%3Atoolbar=yes%2Ctop](https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProduct?%3AshowVizHome=no&%3Aembed=true&%3Atoolbar=yes%2Ctop)

For testing the validity of XPath expressions to extract content using Selenium, we can use JavaScript code

For example, on the page https://www.cnn.com/election/2024/results/florida

```js
counties_link = document.evaluate('//a[text()="See county results"]', document, null, XPathResult.FIRST_ORDERED_NODE_TYPE, null);
counties_link.singleNodeValue;
```

After navigating to the County results, use additional JavaScript code to find the table we want to access, and extract the values from it that we need

For example, on the page https://www.cnn.com/election/2024/results/florida/president

```js
curr_state_votes = []

rows = document.evaluate('//h2[text()="Results by county"]/following-sibling::div/table/tbody/tr', document, null, XPathResult.ORDERED_NODE_SNAPSHOT_TYPE, null);

for(r=0, r<rows.snapshotLength, r++) {
    curr_row = rows.snapshotItem(r);
    county_name = document.evaluate('th/div/text()', curr_row, null, XPathResult.FIRST_ORDERED_NODE_TYPE, null).singleNodeValue.data;
    curr_cells = document.evaluate('td', curr_row, null, XPathResult.ORDERED_NODE_SNAPSHOT_TYPE, null);

    curr_dem = curr_cells.snapshotItem(0);
    curr_dem_divs = document.evaluate('div', curr_dem, null, XPathResult.ORDERED_NODE_SNAPSHOT_TYPE, null);
    curr_dem_percent = (parseFloat(curr_dem_divs.snapshotItem(0).innerText) / 100.0).toFixed(3);
    curr_dem_votecnt = parseInt(curr_dem_divs.snapshotItem(1).innerText.replace(",", ""));

    curr_rep = curr_cells.snapshotItem(curr_cells.snapshotLength - 1);
    curr_rep_divs = document.evaluate('div', curr_rep, null, XPathResult.ORDERED_NODE_SNAPSHOT_TYPE, null);
    curr_rep_percent = (parseFloat(curr_rep_divs.snapshotItem(0).innerText) / 100.0).toFixed(3);
    curr_rep_votecnt = parseInt(curr_rep_divs.snapshotItem(1).innerText.replace(",", ""));

    curr_state_votes.push({statename: "florida", countyname: county_name, dempercent: curr_dem_percent, demvotecnt: curr_dem_votecnt, reppercent: curr_rep_percent, repvotecnt: curr_rep_votecnt});
}
```

In [3]:
# Import the libraries we will use to scrape symbol data
import io
import os
import glob
import re
import csv

from chromedriver_py import binary_path # This gets the path to the chromedriver executable
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

import time
from datetime import datetime, timedelta, timezone
import pandas as pd

# For plotting USA map
import plotly.express as px
import kaleido

The above JavaScript also will work only partially because the trade website uses Tableau for generating these reports and the value of the exports, although displayed, cannot be obtained by using XPath (possibly generated on the fly with JavaScript?). So, we can get the ranking of export destination, but without the value of the exports.

However, we can obtain the values of imports and exports interactively if not programmatically. Specifically, we simply need to browse to:

[https://www.trade.gov/data-visualization/tradestats-express-state-trade-partners](https://www.trade.gov/data-visualization/tradestats-express-state-trade-partners)

and by setting the filter to the most recent year available, and swapping between World, and Canada for countries while also swapping between exports and Imports, we can get all the data we need in CSV format.

In [4]:
# We also need a translation dictionary from state name to state abbreviation (for mapping)

dic_states = {
	  "Alabama": "AL"
	, "Alaska": "AK"
	, "Arizona": "AZ"
	, "Arkansas": "AR"
	, "California": "CA"
	, "Colorado": "CO"
	, "Connecticut": "CT"
	, "Delaware": "DE"
	, "District of Columbia": "DC"
	, "Florida": "FL"
	, "Georgia": "GA"
	, "Hawaii": "HI"
	, "Idaho": "ID"
	, "Illinois": "IL"
	, "Indiana": "IN"
	, "Iowa": "IA"
	, "Kansas": "KS"
	, "Kentucky": "KY"
	, "Louisiana": "LA"
	, "Maine": "ME"
	, "Maryland": "MD"
	, "Massachusetts": "MA"
	, "Michigan": "MI"
	, "Minnesota": "MN"
	, "Mississippi": "MS"
	, "Missouri": "MO"
	, "Montana": "MT"
	, "Nebraska": "NE"
	, "Nevada": "NV"
	, "New Hampshire": "NH"
	, "New Jersey": "NJ"
	, "New Mexico": "NM"
	, "New York": "NY"
	, "North Carolina": "NC"
	, "North Dakota": "ND"
	, "Ohio": "OH"
	, "Oklahoma": "OK"
	, "Oregon": "OR"
	, "Pennsylvania": "PA"
	, "Puerto Rico": "PR"
	, "Rhode Island": "RI"
	, "South Carolina": "SC"
	, "South Dakota": "SD"
	, "Tennessee": "TN"
	, "Texas": "TX"
	, "Utah": "UT"
	, "Vermont": "VT"
	, "Virginia": "VA"
	, "Washington": "WA"
	, "West Virginia": "WV"
	, "Wisconsin": "WI"
	, "Wyoming": "WY"
}

In [5]:
# Get a listing of the CSV files we previously downloaded interactively as described above

def parse_currency(value):
    if pd.isna(value) or value == '':
        return 0
    # Strip symbols and convert to whole number
    clean_value = str(value).replace('$', '').replace(',', '')
    return int(float(clean_value)) # float conversion handles any sneaky trailing decimals

df_imp_exp_values = pd.DataFrame(
    columns={
          "State": str
        , "Flow": str
        , f"Canada_Trade_{DATA_YEAR}": int
        , f"Canada_Percent_{DATA_YEAR}": float
    }
)

for direc in ["Export", "Import"]:

    df_values = pd.DataFrame(
        columns={
            "State": str
            , "Flow": str
            , "Country": str
            , f"{DATA_YEAR}": int
        }
    )

    for markt in ["World", "Canada"]:
        csv_files = glob.glob(f"*{direc}*{markt}*{DATA_YEAR}.csv")
        if len(csv_files) > 0:
            csv_file = csv_files[0]
            print(f"Processing file '{csv_file}'")
            df_temp = pd.read_csv(csv_file, delimiter="\t", usecols=["State", "Flow", "Country", f"{DATA_YEAR}"], converters={f"{DATA_YEAR}": parse_currency}, encoding="utf-16")
            df_temp = df_temp[df_temp["State"].isin(dic_states.keys())]

            if df_values.shape[0] == 0:
                df_values = df_temp.copy()
            else:
                df_values = pd.merge(df_values, df_temp, on=["State", "Flow"], how="inner", suffixes=["_df1", "_df2"])
                df_values[f"Canada_Trade_{DATA_YEAR}"] = df_values[f"{DATA_YEAR}_df1"]
                df_values[f"Canada_Percent_{DATA_YEAR}"] = (df_values[f"{DATA_YEAR}_df2"] / df_values[f"Canada_Trade_{DATA_YEAR}"]).round(4)
                df_values.drop(columns=[f"Country_df1", f"Country_df2", f"{DATA_YEAR}_df1", f"{DATA_YEAR}_df2"], inplace=True)

    if df_imp_exp_values.shape[0] == 0:
        df_imp_exp_values = df_values.copy()
    else:
        df_imp_exp_values = pd.concat([df_imp_exp_values, df_values], ignore_index=True)

# df_imp_exp_values[(df_imp_exp_values["State"]=="Maine") & (df_imp_exp_values["Flow"]=="Exports")]


Processing file 'Table - Product Export Trade - World - 2025.csv'
Processing file 'Table - Product Export Trade - Canada - 2025.csv'
Processing file 'Table - Product Import Trade - World - 2025.csv'
Processing file 'Table - Product Import Trade - Canada - 2025.csv'


### Top 10 states by export percentage to Canada of all their exports worldwide

In [6]:
df_imp_exp_values[df_imp_exp_values["Flow"]=="Exports"].sort_values(by="Canada_Percent_2025", ascending=False).head(10)

,State,Flow,Canada_Trade_2025,Canada_Percent_2025
35,North Dakota,Exports,8677,0.8060
47,Montana,Exports,2165,0.4656
50,South Dakota,Exports,1856,0.4585
42,West Virginia,Exports,4648,0.4410
46,Maine,Exports,3193,0.4122
43,Idaho,Exports,4609,0.3864
8,Michigan,Exports,60328,0.3853
25,Missouri,Exports,19637,0.3414
10,Ohio,Exports,56514,0.3234
48,Vermont,Exports,2059,0.3084


### Top 10 states by import percentage to Canada of all their imports worldwide

In [7]:
df_imp_exp_values[df_imp_exp_values["Flow"]=="Imports"].sort_values(by="Canada_Percent_2025", ascending=False).head(10)

,State,Flow,Canada_Trade_2025,Canada_Percent_2025
94,Montana,Imports,6292,0.9137
97,Vermont,Imports,3964,0.7003
95,Maine,Imports,5995,0.6869
98,North Dakota,Imports,3925,0.6757
103,Wyoming,Imports,988,0.4868
83,Oklahoma,Imports,17063,0.4591
99,Alaska,Imports,3141,0.4072
96,West Virginia,Imports,4885,0.3773
101,South Dakota,Imports,1537,0.3604
72,Minnesota,Imports,37261,0.3355


In [8]:
# Let's check where the chromedriver executable was installed

binary_path

'C:\\Users\\cesin\\AppData\\Roaming\\Python\\Python311\\site-packages\\chromedriver_py\\chromedriver_win64.exe'

In [9]:
CHROMEDRIVER_USER_DIR_PATH = "/".join(binary_path.split("\\")[:-1])
CHROMEDRIVER_USER_DIR_PATH

'C:/Users/cesin/AppData/Roaming/Python/Python311/site-packages/chromedriver_py'

In [10]:
# Setup our timeouts

timeout = 30
pause = 3

In [11]:
chrome_options = Options()
# See list of options/switches for Chrome here:
# https://peter.sh/experiments/chromium-command-line-switches/
#chrome_options.add_argument("--headless") # Run with any UI
chrome_options.add_argument("--start-maximized") # Maximize the window to fit to the screen
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--user-data-dir=" + CHROMEDRIVER_USER_DIR_PATH)
#driver = webdriver.Chrome(executable_path=CHROMEDRIVER_EXECUTABLE_PATH, chrome_options=chrome_options)
webdriver.ChromeOptions = chrome_options
driver = webdriver.Chrome()
driver.set_window_size(1920, 1080)

In [12]:
# Set the base URL

url = "https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProduct?%3AshowVizHome=no&%3Aembed=true&%3Atoolbar=yes%2Ctop"

# New, updated link "TradeStats Express - Trade by Partner by State":
url = "https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/TradebyPartnerbyState?%3Atoolbar=yes%2Ctop&%3AshowVizHome=no&%3Aembed=true"
# or is it "TradeStats Express - State Trade by Products":
url = "https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyProducts?%3Atoolbar=yes%2Ctop&%3AshowVizHome=no&%3Aembed=true"
# No - this is definitely the one we want i.e. "TradeStats Express - State Trade by Partners":
url = "https://tsereports.trade.gov/views/StateAnnualDashboardLive_16421978287720/StateTradebyPartners?%3Atoolbar=yes%2Ctop&%3AshowVizHome=no&%3Aembed=true"

In [13]:
# Load the initial web page from the base URL

wait = WebDriverWait(driver, timeout)
driver.get(url)
#WebDriverWait(driver=driver, timeout=timeout).until(EC.visibility_of_element_located((By.XPATH, '//div[@class="tab-tvView tvimagesNS" and @role="region"]')))
WebDriverWait(driver=driver, timeout=timeout).until(EC.visibility_of_element_located((By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_2"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')))

<selenium.webdriver.remote.webelement.WebElement (session="7d809ada2b68dee7abb59e77470b91f8", element="f.D4BFC0B21F59667A9E37B974DBDF1F20.d.7A4ADC4315CB75EFA5A9BF57782D7C90.e.21")>

In [14]:
years_since = driver.find_element(By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_2"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')
years_since

<selenium.webdriver.remote.webelement.WebElement (session="7d809ada2b68dee7abb59e77470b91f8", element="f.D4BFC0B21F59667A9E37B974DBDF1F20.d.7A4ADC4315CB75EFA5A9BF57782D7C90.e.21")>

In [15]:
# Display the oldest year in the data being displayed
#  
years_since.text

'2019'

In [16]:
# Click on the year combobox to change to the most recent year (as of this writing, 2023)

years_since.click()

In [17]:
# Bty clicking ont he current year, it will open the combobox allowing us to select the target year

years_combo = driver.find_element(By.XPATH, '//div[@role="menu" and contains(@aria-label, "Start Year:")]')

In [18]:
# Locate the combobox item representing the year 2023

year_curr = years_combo.find_element(By.XPATH, f'./div[@class="tabMenuContent"]//span[text() = "{DATA_YEAR}"]')
year_curr.click()

# Force Python to sleep (note that this is necessary as the drive wait doesn't pause execution)
time.sleep(pause)

In [19]:
# Set up our dictionary for export countries by state
state_exports = {}

# Loop until the last state is clicked
while True:

    # Get the list of countries being displayed and choose the 1st item (the 0th item is a header)
    top_countries = driver.find_elements(By.XPATH, '//div[@class="tab-tvYLabel tvimagesNS"]/div[@class="tab-ReactView"]/div[@class="tab-vizLeftSceneMargin"]//div[@class="tab-vizHeaderWrapper"]')
    curr_export_dest = top_countries[1].text

    # Delete the overlaid div that prevents the element from being clicked
    delete_elements = driver.find_elements(By.XPATH, '//div[@class="wcGlassPane"]') # id="loadingGlassPane"
    if len(delete_elements) > 0:
        driver.execute_script("""
            var element = arguments[0];
            element.parentNode.removeChild(element);
        """, delete_elements[0])

    # Wait until the JavaScript code deletes the element
    #wait = WebDriverWait(driver, pause)
    time.sleep(pause)

    # Locate the currently displayed state and click it (this will open the combobox of states again)
    WebDriverWait(driver=driver, timeout=timeout).until(EC.element_to_be_clickable((By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_1"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')))
    curr_state = driver.find_element(By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_1"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')
    curr_state.click()
    
    # Locate the State combobox
    state_combo = driver.find_element(By.XPATH, '//div[@role="menu" and contains(@aria-label, "State:")]')

    # Save the name of the currently selected state and use it to add an entry into the dictionary
    curr_state_text = curr_state.text
    state_exports[dic_states[curr_state_text]] = [curr_state_text, curr_export_dest]

    # Now find the next element in the combobox based on the currently selected coombobox element (use following-sibling to navigate to the next one)
    state_curr = state_combo.find_element(By.XPATH, f'./div[@class="tabMenuContent"]//span[text() = "{curr_state.text}"]')
    state_next = state_curr.find_elements(By.XPATH, f'../../following-sibling::*')

    # Break out of the loop once the last state has been reached and there are no "following-siblings"
    if len(state_next) < 1:
        state_curr.click() # Close the combo box
        break

    # Since there are still some states remaining that we haven't visited yet, click on the next state name
    state_next[0].click()

    # Pause to allow the screen to reload with new data pertinent to the state
    time.sleep(pause)

In [20]:
# Now, let's do the imports

# Click the radio button to change from Exports to Imports
rdo_imports = driver.find_element(By.XPATH, '//a[text()="Imports"]/preceding-sibling::input[@type="radio"]')
rdo_imports.click()
time.sleep(pause)

In [21]:
# Open the combo box of state names
state_control = driver.find_element(By.XPATH, '//h3[@title="State:"]/../../../div/div[@class="PCContent"]/span')
state_control.click()
time.sleep(pause)

In [22]:
 # Locate the State combobox
state_combo = driver.find_element(By.XPATH, '//div[@role="menu" and contains(@aria-label, "State:")]')

In [23]:
# Get the list of states, and click on the first one (Alabama)
all_states = state_combo.find_elements(By.XPATH, f'./div[@class="tabMenuContent"]//span')
print(len(all_states))
all_states[0].click()
time.sleep(pause)

52


In [24]:
# Set up our dictionary for import countries by state
state_imports = {}

# Loop until the last state is clicked
while True:

    # Get the list of countries being displayed and choose the 1st item (the 0th item is a header)
    top_countries = driver.find_elements(By.XPATH, '//div[@class="tab-tvYLabel tvimagesNS"]/div[@class="tab-ReactView"]/div[@class="tab-vizLeftSceneMargin"]//div[@class="tab-vizHeaderWrapper"]')
    curr_import_dest = top_countries[1].text

    # Delete the overlaid div that prevents the element from being clicked
    delete_elements = driver.find_elements(By.XPATH, '//div[@class="wcGlassPane"]') # id="loadingGlassPane"
    if len(delete_elements) > 0:
        driver.execute_script("""
            var element = arguments[0];
            element.parentNode.removeChild(element);
        """, delete_elements[0])

    # Wait until the JavaScript code deletes the element
    #wait = WebDriverWait(driver, pause)
    time.sleep(pause)

    # Locate the currently displayed state and click it (this will open the combobox of states again)
    WebDriverWait(driver=driver, timeout=timeout).until(EC.element_to_be_clickable((By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_1"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')))
    curr_state = driver.find_element(By.XPATH, '//div[@id="tableau_base_widget_ParameterControl_1"]//div[@class="tabComboBoxNameContainer tab-ctrl-formatted-fixedsize"]/span')
    curr_state.click()
    
    # Locate the State combobox
    state_combo = driver.find_element(By.XPATH, '//div[@role="menu" and contains(@aria-label, "State:")]')

    # Save the name of the currently selected state and use it to add an entry into the dictionary
    curr_state_text = curr_state.text
    state_imports[dic_states[curr_state_text]] = [curr_state_text, curr_import_dest]

    # Now find the next element in the combobox based on the currently selected coombobox element (use following-sibling to navigate to the next one)
    state_curr = state_combo.find_element(By.XPATH, f'./div[@class="tabMenuContent"]//span[text() = "{curr_state.text}"]')
    state_next = state_curr.find_elements(By.XPATH, f'../../following-sibling::*')

    # Break out of the loop once the last state has been reached and there are no "following-siblings"
    if len(state_next) < 1:
        state_curr.click() # Close the combo box
        break

    # Since there are still some states remaining that we haven't visited yet, click on the next state name
    state_next[0].click()

    # Pause to allow the screen to reload with new data pertinent to the state
    time.sleep(pause)

In [25]:
# Let's look at the data we collected for export countries

list(state_exports.items())[:10]

[('AL', ['Alabama', 'Canada']),
 ('AK', ['Alaska', 'South Korea']),
 ('AZ', ['Arizona', 'Mexico']),
 ('AR', ['Arkansas', 'Mexico']),
 ('CA', ['California', 'Mexico']),
 ('CO', ['Colorado', 'Mexico']),
 ('CT', ['Connecticut', 'Canada']),
 ('DE', ['Delaware', 'Canada']),
 ('DC', ['District of Columbia', 'Saudi Arabia']),
 ('FL', ['Florida', 'Brazil'])]

In [26]:
# Let's look at the data we collected for import countries

list(state_imports.items())[:10]

[('AL', ['Alabama', 'Mexico']),
 ('AK', ['Alaska', 'Canada']),
 ('AZ', ['Arizona', 'Taiwan']),
 ('AR', ['Arkansas', 'Canada']),
 ('CA', ['California', 'China']),
 ('CO', ['Colorado', 'Canada']),
 ('CT', ['Connecticut', 'Canada']),
 ('DE', ['Delaware', 'Mexico']),
 ('DC', ['District of Columbia', 'Australia']),
 ('FL', ['Florida', 'China'])]

In [27]:
driver.quit()

In [28]:
df_exports = pd.DataFrame.from_dict(state_exports, columns=["State", "Destination"], orient="index")
#df_exports = pd.DataFrame.from_dict(abbrev_exports, columns=["State", "Destination"], orient="index")
df_exports.index.names = ["StateAbbr"]
df_exports.reset_index(drop=False, inplace=True)
# Since Puerto Rico doesn't appear on the map, we will drop it from the dataframe
df_exports = df_exports[df_exports["StateAbbr"] != "PR"]
df_exports.head(10)

,StateAbbr,State,Destination
0,AL,Alabama,Canada
1,AK,Alaska,South Korea
2,AZ,Arizona,Mexico
3,AR,Arkansas,Mexico
4,CA,California,Mexico
5,CO,Colorado,Mexico
6,CT,Connecticut,Canada
7,DE,Delaware,Canada
8,DC,District of Columbia,Saudi Arabia
9,FL,Florida,Brazil


In [29]:
df_export_labels = pd.merge(df_exports[df_exports["Destination"]=="Canada"], df_imp_exp_values[df_imp_exp_values["Flow"]=="Exports"], on=["State"], how="inner")
df_export_labels.drop(columns=["Destination", "Flow"], inplace=True)
df_export_labels["Label_Text"] = df_export_labels.apply(
      lambda row: "${:,.0f}<br>{:.2%}".format(row[f"Canada_Trade_{DATA_YEAR}"], row[f"Canada_Percent_{DATA_YEAR}"])
    , axis=1
)
df_export_labels

,StateAbbr,State,Canada_Trade_2025,Canada_Percent_2025,Label_Text
0,AL,Alabama,23671,0.1820,"$23,671<br>18.20%"
1,CT,Connecticut,17802,0.1236,"$17,802<br>12.36%"
2,DE,Delaware,5507,0.1407,"$5,507<br>14.07%"
3,GA,Georgia,60290,0.1141,"$60,290<br>11.41%"
4,ID,Idaho,4609,0.3864,"$4,609<br>38.64%"
5,IL,Illinois,80195,0.2256,"$80,195<br>22.56%"
6,IA,Iowa,16310,0.3040,"$16,310<br>30.40%"
7,ME,Maine,3193,0.4122,"$3,193<br>41.22%"
8,MD,Maryland,16561,0.1316,"$16,561<br>13.16%"
9,MI,Michigan,60328,0.3853,"$60,328<br>38.53%"


In [30]:
fig = px.choropleth(
      df_exports
    , locations = "StateAbbr"
    , locationmode = "USA-states"
    , color = "Destination"
    , scope = "usa"
    , hover_data = {"StateAbbr": False, "State": True, "Destination": True}
    #, labels = {"Destination": "Target", "State":""}
    , title = "Primary Export Country by State"
    #, width = 1024
    #, height = 768
    , category_orders = {"Destination": sorted(pd.unique(df_exports["Destination"]))}
    #, color_discrete_sequence = px.colors.qualitative.Alphabet[1:]
    #, color_discrete_sequence = px.colors.qualitative.Light24
    , color_discrete_map = SHARED_COLOR_MAP
)

fig.add_scattergeo(
      locations=df_export_labels["StateAbbr"]
    , locationmode="USA-states"
    , text=df_export_labels["Label_Text"]
    , mode="text"
    , textfont=dict(
          size=10
        , color="black"
        , family="Arial-Bold"
    )
    , showlegend=False  # Hides the scatter trace from the legend
)

fig.update_layout(
      width = 1000
    , height = 500
    , margin = {"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()
fig.write_image("images/export_countries_by_state.png")

![export_countries_by_state.png](./images/export_countries_by_state.png)

In [31]:
df_imports = pd.DataFrame.from_dict(state_imports, columns=["State", "Source"], orient="index")
#df_imports = pd.DataFrame.from_dict(abbrev_imports, columns=["State", "Source"], orient="index")
df_imports.index.names = ["StateAbbr"]
df_imports.reset_index(drop=False, inplace=True)
# Since Puerto Rico doesn't appear on the map, we will drop it from the dataframe
df_imports = df_imports[df_imports["StateAbbr"] != "PR"]
df_imports.head(10)

,StateAbbr,State,Source
0,AL,Alabama,Mexico
1,AK,Alaska,Canada
2,AZ,Arizona,Taiwan
3,AR,Arkansas,Canada
4,CA,California,China
5,CO,Colorado,Canada
6,CT,Connecticut,Canada
7,DE,Delaware,Mexico
8,DC,District of Columbia,Australia
9,FL,Florida,China


In [32]:
df_import_labels = pd.merge(df_imports[df_imports["Source"]=="Canada"], df_imp_exp_values[df_imp_exp_values["Flow"]=="Imports"], on=["State"], how="inner")
df_import_labels.drop(columns=["Source", "Flow"], inplace=True)
df_import_labels["Label_Text"] = df_import_labels.apply(
      lambda row: "${:,.0f}<br>{:.2%}".format(row[f"Canada_Trade_{DATA_YEAR}"], row[f"Canada_Percent_{DATA_YEAR}"])
    , axis=1
)
df_import_labels

,StateAbbr,State,Canada_Trade_2025,Canada_Percent_2025,Label_Text
0,AK,Alaska,3141,0.4072,"$3,141<br>40.72%"
1,AR,Arkansas,6672,0.1596,"$6,672<br>15.96%"
2,CO,Colorado,16374,0.3250,"$16,374<br>32.50%"
3,CT,Connecticut,24026,0.2380,"$24,026<br>23.80%"
4,IL,Illinois,210871,0.2765,"$210,871<br>27.65%"
5,IA,Iowa,11973,0.2656,"$11,973<br>26.56%"
6,KS,Kansas,13918,0.1418,"$13,918<br>14.18%"
7,ME,Maine,5995,0.6869,"$5,995<br>68.69%"
8,MA,Massachusetts,42840,0.2326,"$42,840<br>23.26%"
9,MN,Minnesota,37261,0.3355,"$37,261<br>33.55%"


In [33]:
fig = px.choropleth(
      df_imports
    , locations = "StateAbbr"
    , locationmode = "USA-states"
    , color = "Source"
    , scope = "usa"
    , hover_data = {"StateAbbr": False, "State": True, "Source": True}
    #, labels = {"Source": "Target", "State":""}
    , title = "Primary Import Country by State"
    #, width = 1024
    #, height = 768
    , category_orders = {"Source": sorted(pd.unique(df_imports["Source"]))}
    #, color_discrete_sequence = px.colors.qualitative.Alphabet[1:]
    #, color_discrete_sequence = px.colors.qualitative.Light24
    , color_discrete_map = SHARED_COLOR_MAP
)

fig.add_scattergeo(
      locations=df_import_labels["StateAbbr"]
    , locationmode="USA-states"
    , text=df_import_labels["Label_Text"]
    , mode="text"
    , textfont=dict(
          size=10
        , color="black"
        , family="Arial-Bold"
    )
    , showlegend=False  # Hides the scatter trace from the legend
)

fig.update_layout(
      width = 1000
    , height = 500
    , margin = {"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()
fig.write_image("images/import_countries_by_state.png")

![import_countries_by_state.png](./images/import_countries_by_state.png)